In [0]:
bookings_df = spark.table("bronze_bookings")
flights_df = spark.table("bronze_flights")
preferences_flat = spark.table("bronze_preferences")

In [0]:
from pyspark.sql.functions import col,when

bookings_df = bookings_df.withColumn(
    "revenue",
    col("ticket_price")
)

In [0]:
bookings_df = bookings_df.withColumn(
    "price_band",
    when(col("ticket_price") > 20000,"Premium")
    .when(col("ticket_price") > 10000,"Standard")
    .otherwise("Budget")
)

In [0]:
flights_df = flights_df.withColumn(
    "delay_flag",
    when(col("status")=="Delayed","Yes")
    .otherwise("No")
)

In [0]:
print("Bookings:", bookings_df.count())
print("Flights:", flights_df.count())
print("Preferences:", preferences_flat.count())

Bookings: 20
Flights: 16
Preferences: 16


In [0]:
display(bookings_df)
display(flights_df)

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date,revenue,price_band
B1001,F101,Rahul Sharma,Economy,8500.0,2026-06-01,8500.0,Budget
B1002,F101,Priya Reddy,Business,22000.0,2026-06-01,22000.0,Premium
B1003,F102,Amit Kumar,Economy,9000.0,2026-06-02,9000.0,Budget
B1004,F103,Sneha Patel,Premium Economy,15000.0,2026-06-02,15000.0,Standard
B1005,F104,Farhan Ali,Economy,7500.0,2026-06-03,7500.0,Budget
B1006,F105,Neha Singh,Business,25000.0,2026-06-03,25000.0,Premium
B1007,F106,Arjun Verma,Economy,10000.0,2026-06-04,10000.0,Budget
B1008,F107,Meera Nair,Premium Economy,17000.0,2026-06-04,17000.0,Standard
B1009,F108,Kiran Rao,Economy,9500.0,2026-06-05,9500.0,Budget
B1010,F109,Nisha Reddy,Business,28000.0,2026-06-05,28000.0,Premium


flight_id,airline,from_city,to_city,duration,status,delay_flag
"""flight_id",airline,from_city,to_city,duration,"status""",No
"""F101",Indigo,Hyderabad,Delhi,140,"On Time""",No
"""F102",Air India,Mumbai,Chennai,120,"Delayed""",No
"""F103",Vistara,Bangalore,Hyderabad,90,"On Time""",No
"""F104",Indigo,Delhi,Mumbai,130,"Cancelled""",No
"""F105",Air India,Chennai,Bangalore,80,"On Time""",No
"""F106",Akasa,Pune,Delhi,150,"Delayed""",No
"""F107",Vistara,Hyderabad,Kolkata,160,"On Time""",No
"""F108",Indigo,Mumbai,Hyderabad,110,"On Time""",No
"""F109",Akasa,Delhi,Chennai,145,"Delayed""",No


In [0]:
%sql
DROP TABLE IF EXISTS booking_master;

In [0]:
journey_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("booking_master")

In [0]:
journey_df = bookings_df.join(
    flights_df,
    "flight_id"
).join(
    preferences_flat,
    "passenger_name",
    "left"
)

display(journey_df)

passenger_name,flight_id,booking_id,travel_class,ticket_price,booking_date,revenue,price_band,airline,from_city,to_city,duration,status,delay_flag,meal,seat,extra_baggage


In [0]:
test_join = bookings_df.join(
    flights_df,
    "flight_id"
)

print("Flight Join Count:", test_join.count())
display(test_join)

Flight Join Count: 0


flight_id,booking_id,passenger_name,travel_class,ticket_price,booking_date,revenue,price_band,airline,from_city,to_city,duration,status,delay_flag


In [0]:
bookings_df.select("flight_id").distinct().show(20, False)

+---------+
|flight_id|
+---------+
|F101     |
|F102     |
|F103     |
|F104     |
|F105     |
|F106     |
|F107     |
|F108     |
|F109     |
|F110     |
|F111     |
|F112     |
|F113     |
|F114     |
|F115     |
+---------+



In [0]:
flights_df.select("flight_id").distinct().show(20, False)

+----------+
|flight_id |
+----------+
|"flight_id|
|"F101     |
|"F102     |
|"F103     |
|"F104     |
|"F105     |
|"F106     |
|"F107     |
|"F108     |
|"F109     |
|"F110     |
|"F111     |
|"F112     |
|"F113     |
|"F114     |
|"F115     |
+----------+



---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8081618927852722>, line 3
      1 from pyspark.sql.functions import split,col,regexp_replace
----> 3 raw_flights = spark.read.text(flights_file)
      5 clean_flights = raw_flights.select(
      6     regexp_replace(col("value"), '"', "").alias("value")
      7 )
      9 flights_df = clean_flights.select(
     10     split(col("value"), ",").getItem(0).alias("flight_id"),
     11     split(col("value"), ",").getItem(1).alias("airline"),
   (...)
     15     split(col("value"), ",").getItem(5).alias("status")
     16 )

NameError: name 'flights_file' is not defined